# Cài đặt lại thuật toán FPClose bằng Julia

**Đồ án môn Khai thác dữ liệu và ứng dụng.** Nhóm cài đặt lại từ đầu thuật toán **FPClose**, khai thác tập phổ biến đóng (closed frequent itemsets) dựa trên cấu trúc FP-tree, theo mô tả trong bài báo gốc:

> G. Grahne và J. Zhu, "Fast Algorithms for Frequent Itemset Mining Using FP-Trees", *IEEE Transactions on Knowledge and Data Engineering*, vol. 17, no. 10, pp. 1347–1362, 2005. DOI: [10.1109/TKDE.2005.166](https://doi.org/10.1109/TKDE.2005.166)

Notebook này trình bày toàn bộ quy trình của đồ án: cài đặt môi trường, chuẩn bị dữ liệu, kiểm thử, đối chiếu kết quả với SPMF, thực nghiệm đánh giá và các ứng dụng thực tế.

**Thành viên nhóm:**
- Cao Hoàng Long — 24127442
- Nguyễn Khánh Linh — 24127197
- Hoàng Yến Trang — 24127133

**Tài nguyên:**
- Mã nguồn: https://github.com/DRAGON-1312/FPClose
- Bộ dữ liệu benchmark đã chuẩn hóa (Hugging Face): https://huggingface.co/datasets/hytrang/fpclose-fim-benchmarks

**Cách thực thi:** notebook được thiết kế để chạy trên Google Colab thông qua *Runtime → Run all*, với kernel **Python 3**; các bước sử dụng Julia được gọi qua lệnh shell `!julia ...`. Các bước cài đặt, tải mã nguồn và tải dữ liệu sẽ tự động được bỏ qua nếu đã thực hiện trước đó, nhờ vậy notebook có thể chạy lại nhiều lần. Mỗi mục tự kiểm tra các điều kiện cần thiết (dữ liệu, Java, thư viện Python) và chỉ phụ thuộc vào mục 1, do đó lỗi phát sinh ở một mục không ảnh hưởng đến các mục còn lại.

**Nội dung:**
1. Cài đặt môi trường
2. Chuẩn bị dữ liệu
3. Kiểm thử tự động (unit test)
4. Minh họa FPClose qua dòng lệnh
5. Đối chiếu với SPMF
6. Thực nghiệm
7. Ứng dụng thực tế (Chương 3)
8. Kết luận

## 1. Cài đặt môi trường

Notebook được thực thi trên **Google Colab** (hệ điều hành Ubuntu, sử dụng CPU, không yêu cầu GPU) với kernel **Python 3**. Môi trường thực thi gồm các thành phần sau:

| Thành phần | Phiên bản / nguồn | Mục đích sử dụng |
|---|---|---|
| Julia | **1.13.4**, bản phát hành chính thức (tải từ julialang.org, giải nén vào `/usr`) | Thực thi cài đặt FPClose của nhóm (`src/`, `demo.jl`, `scripts/`) |
| Gói Julia | Chỉ thư viện chuẩn của Julia (`Project.toml` không khai báo gói ngoài); `Pkg.instantiate()` và `Pkg.precompile()` chuẩn bị môi trường và biên dịch trước gói `FPClose` | Mã nguồn, các script và bộ kiểm thử |
| Java | **Temurin 25**, được `scripts/reference_check.jl` tự động tải vào `tools/` | Chỉ dùng để chạy SPMF phục vụ đối chiếu kết quả |
| SPMF | `spmf.jar`, được tự động tải vào `tools/` | Cài đặt tham chiếu để so sánh, không tham gia vào thuật toán của nhóm |
| Python | Python 3 có sẵn trên Colab, cùng `pandas` (mục 7.1 dùng thêm `openpyxl`, `jinja2`) | Hiển thị bảng và hình; biến đổi dữ liệu thành giao dịch ở mục 7 |
| Dữ liệu | Hugging Face `hytrang/fpclose-fim-benchmarks` (mục 2); dữ liệu của mục 7 được tải trong từng mục | 9 bộ benchmark; các tập con của Retail và CSDL tổng hợp được `scripts/data_setup.jl` sinh tại chỗ |

Phiên bản Julia được chọn trùng với phiên bản trên máy dùng để đo kết quả chính thức (chi tiết tại `results/ENVIRONMENT.md`).

Ô cấu hình bên dưới khai báo địa chỉ repository (`REPO_URL`), nhánh mã nguồn (`BRANCH`), phiên bản Julia (`JULIA_VERSION`), thư mục clone (`REPO_DIR`) và nhãn máy ghi vào các số đo minh họa ở mục 6a (`MACHINE_LABEL`). Bên cạnh đó, ô này định nghĩa một số hàm tiện ích được các mục sau sử dụng để tự chuẩn bị điều kiện thực thi:
- `sh(cmd)`: thực thi lệnh shell tại thư mục gốc của repository;
- `ensure_data([...])`: tải bổ sung các bộ dữ liệu còn thiếu;
- `ensure_java()`: chuẩn bị Java 25 và `spmf.jar` phục vụ việc chạy SPMF;
- `need_py(...)`: cài đặt các thư viện Python còn thiếu.

In [ ]:
# ===== CẤU HÌNH =====
REPO_URL = 'https://github.com/DRAGON-1312/FPClose.git'
BRANCH = 'main'           # nhánh chính, đã có đủ demo.jl và market_basket.py (mục 7.1); đổi nhánh khác để chạy thử trước khi merge
JULIA_VERSION = '1.13.4'   # cùng phiên bản với máy đo chính thức (results/ENVIRONMENT.md)
REPO_DIR = '/content/FPClose'
MACHINE_LABEL = 'colab'    # nhãn cột machine cho các số đo minh họa trên Colab

import os, glob, time, importlib
os.environ['FPCLOSE_MACHINE'] = MACHINE_LABEL
T_START = time.time()

# ---- Hàm tiện ích: mỗi mục sau tự kiểm tra/chuẩn bị điều kiện của mình bằng các hàm này ----
def sh(cmd):
    """Chạy lệnh shell (từ gốc repo nếu đã clone), output hiện ngay trong ô."""
    if os.path.isdir(REPO_DIR):
        os.chdir(REPO_DIR)
    get_ipython().system(cmd)

def ensure_data(names):
    """Bảo đảm có data/benchmark/<name>.txt; thiếu thì gọi scripts/data_setup.jl --only <các file thiếu>."""
    path = lambda n: os.path.join(REPO_DIR, 'data', 'benchmark', f'{n}.txt')
    missing = [n for n in names if not os.path.exists(path(n))]
    if missing:
        print('Thiếu dữ liệu', missing, '-> chạy scripts/data_setup.jl --only', ','.join(missing))
        sh('julia --project=. scripts/data_setup.jl --only ' + ','.join(missing))
    ok = all(os.path.exists(path(n)) for n in names)
    print('Dữ liệu', names, 'sẵn sàng' if ok else 'VẪN THIẾU, bỏ qua bước cần dữ liệu này')
    return ok

def ensure_java():
    """Bảo đảm có Java 25 và tools/spmf.jar bằng setup_tools() có sẵn trong scripts/reference_check.jl
    (trên Linux tự tải Temurin 25), rồi thêm JDK vào PATH: compare_spmf.jl và run_benchmark_grid.jl gọi `java` trên PATH."""
    jdk_bins = lambda: sorted(glob.glob(os.path.join(REPO_DIR, 'tools', 'jdk-25*', 'bin')))
    if not jdk_bins() or not os.path.exists(os.path.join(REPO_DIR, 'tools', 'spmf.jar')):
        sh('''julia --project=. -e 'include("scripts/reference_check.jl"); setup_tools(TOOLS_DIR; with_fimi = false)' ''')
    bins = jdk_bins()
    if bins and bins[-1] not in os.environ['PATH'].split(os.pathsep):
        os.environ['PATH'] = bins[-1] + os.pathsep + os.environ['PATH']
    print('JDK:', bins[-1] if bins else 'không thấy tools/jdk-25*, dùng java có sẵn trên máy')
    sh('java -version 2>&1 | head -n 1')

def need_py(*pkgs):
    """Import được thì thôi, không thì pip install (Colab đã có sẵn pandas, IPython)."""
    for p in pkgs:
        try:
            importlib.import_module(p)
        except ImportError:
            sh(f'pip install -q {p}')

print('Nhánh:', BRANCH, '| Julia', JULIA_VERSION, '| thư mục', REPO_DIR)

In [ ]:
# Cài Julia từ tarball chính thức vào /usr (bỏ qua nếu `julia --version` đã đúng phiên bản).
import subprocess
def julia_version(exe='julia'):
    try:
        return subprocess.run([exe, '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return ''

WANT = f'julia version {JULIA_VERSION}'
if julia_version() == WANT:
    print('Đã có', julia_version(), '-> bỏ qua bước cài')
else:
    if julia_version('/usr/bin/julia') != WANT:
        major_minor = '.'.join(JULIA_VERSION.split('.')[:2])
        url = f'https://julialang-s3.julialang.org/bin/linux/x64/{major_minor}/julia-{JULIA_VERSION}-linux-x86_64.tar.gz'
        print('Tải', url)
        sh(f'curl -fsSL -o /tmp/julia.tar.gz {url} && tar -xzf /tmp/julia.tar.gz -C /usr --strip-components 1 && rm -f /tmp/julia.tar.gz')
    # Colab có sẵn một bản Julia khác ở /usr/local/bin (đứng trước /usr/bin trong PATH): trỏ lệnh `julia` về bản
    # vừa cài, CHỈ khi /usr/bin/julia đúng phiên bản (tải lỗi thì giữ nguyên bản có sẵn).
    if julia_version('/usr/bin/julia') == WANT and os.path.isdir('/usr/local/bin'):
        sh('ln -sf /usr/bin/julia /usr/local/bin/julia')
    if julia_version() != WANT:
        print('CẢNH BÁO: lệnh julia là', julia_version() or '(không tìm thấy)', 'thay vì', JULIA_VERSION)
!julia --version

In [ ]:
# Clone repo (bỏ qua nếu đã có) rồi chuyển sang nhánh BRANCH. Từ đây mọi lệnh chạy từ gốc repo.
![ -d {REPO_DIR}/.git ] || git clone --quiet {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin {BRANCH} && git -C {REPO_DIR} checkout --quiet -B {BRANCH} FETCH_HEAD
%cd {REPO_DIR}
!git log -1 --format='%h %s (%ci)'

In [ ]:
%%time
# Nếu Manifest.toml được sinh bởi phiên bản Julia khác (ví dụ ở lần chạy trước trong cùng phiên),
# xóa đi để Pkg giải lại phụ thuộc bằng đúng JULIA_VERSION; repo chỉ dùng thư viện chuẩn nên việc này an toàn.
import re
manifest = os.path.join(REPO_DIR, 'Manifest.toml')
if os.path.exists(manifest):
    m = re.search(r'julia_version\s*=\s*"([^"]+)"', open(manifest, encoding='utf-8').read())
    if not m or m.group(1) != JULIA_VERSION:
        os.remove(manifest)
        print('Đã xóa Manifest.toml được sinh bởi Julia', m.group(1) if m else '(không rõ)')
# Cài gói theo Project.toml, biên dịch trước và in thông tin phiên bản.
!julia --project=. -e 'using Pkg; Pkg.instantiate(); Pkg.precompile(); using FPClose, InteractiveUtils; versioninfo()'

## 2. Chuẩn bị dữ liệu

Toàn bộ quá trình chuẩn bị dữ liệu được nhóm cài đặt trong `scripts/data_setup.jl`, gồm ba bước:
1. Tải 9 bộ dữ liệu benchmark đã chuẩn hóa (Chess, Mushrooms, Retail, T10I4D100K, Accidents, Connect, Kosarak, Pumsb*, T40I10D100K) từ Hugging Face `hytrang/fpclose-fim-benchmarks`, kiểm tra mã băm SHA-256 nhằm bảo đảm tính toàn vẹn của dữ liệu và đối chiếu thống kê của từng bộ với `data/meta/dataset_statistics.csv`.
2. Sinh 5 tập con của Retail (10%, 25%, 50%, 75% và 100% số giao dịch) tại `data/scalability/`, phục vụ thực nghiệm khả năng mở rộng (scalability).
3. Sinh 6 cơ sở dữ liệu (CSDL) tổng hợp có độ dài giao dịch trung bình tăng dần từ 5 đến 30 tại `data/synthetic/`, phục vụ thực nghiệm về ảnh hưởng của độ dài giao dịch.

Các file sinh ra ở bước 2 và 3 cũng được kiểm tra mã băm để bảo đảm khả năng tái lập. Nguồn tải, mã băm, số giao dịch và seed của các CSDL tổng hợp được lưu trong `data/meta/*.csv`. Trường hợp dữ liệu đã đầy đủ, bước này sẽ được bỏ qua. Các mục sau sử dụng dữ liệu được chuẩn bị tại đây; nếu phát hiện thiếu dữ liệu, mỗi mục sẽ tự tải bổ sung thông qua `data_setup.jl --only ...`. Hai bảng bên dưới trình bày thống kê của các bộ dữ liệu thu được.

In [ ]:
%%time
count_files = lambda pattern: len(glob.glob(os.path.join(REPO_DIR, pattern)))
n_bench = count_files('data/benchmark/*.txt')
n_scal = count_files('data/scalability/*.txt')
n_syn = count_files('data/synthetic/*.txt')
print(f'Hiện có: {n_bench}/9 benchmark, {n_scal}/5 tập con Retail, {n_syn}/6 synthetic')
if n_bench >= 9 and n_scal >= 5 and n_syn >= 6:
    print('Đủ dữ liệu -> bỏ qua data_setup.jl')
else:
    sh('julia --project=. scripts/data_setup.jl')

In [ ]:
need_py('pandas')
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
for p in ['data/meta/dataset_statistics.csv', 'data/meta/synthetic_statistics.csv']:
    full = os.path.join(REPO_DIR, p)
    print('==', p)
    display(pd.read_csv(full)) if os.path.exists(full) else print('  (không có file)')

**Kết quả.** Dữ liệu được chuẩn bị đầy đủ (9 bộ benchmark, 5 tập con của Retail, 6 CSDL tổng hợp) và đều khớp mã băm. Các bộ dữ liệu gồm cả loại dày đặc và loại thưa, phục vụ việc đánh giá thuật toán trên nhiều đặc điểm dữ liệu khác nhau.

## 3. Kiểm thử tự động (unit test)

Mục này thực thi toàn bộ bộ kiểm thử `test/runtests.jl`, bao gồm:
- kiểm tra bộ đọc file định dạng SPMF và các tham số dòng lệnh;
- đối chiếu kết quả với đáp án vét cạn (brute force) trên các CSDL sinh ngẫu nhiên;
- xác nhận mọi tổ hợp bật/tắt các tối ưu đều cho cùng một kết quả;
- so sánh từng itemset và support với output của SPMF đã được lưu sẵn trong `test/fixtures/spmf/` (không cần chạy Java);
- kiểm tra bước khôi phục mọi tập phổ biến từ tập đóng (recovery) và bước sinh luật kết hợp;
- chạy `fpclose` ở minsup (ngưỡng support tối thiểu) 90% trên 9 bộ benchmark đã tải ở mục 2; nhóm test này được bỏ qua nếu mục 2 không tải được dữ liệu.

Dữ liệu ngẫu nhiên trong kiểm thử sử dụng seed cố định nhằm bảo đảm khả năng tái lập. Khi toàn bộ test đều đạt, Julia chỉ hiển thị bảng **Test Summary** dạng thu gọn, gồm số test đạt trên tổng số test; trường hợp có test không đạt, chi tiết lỗi cùng bảng kết quả của từng nhóm test sẽ được hiển thị đầy đủ.

In [ ]:
%%time
!julia --project=. test/runtests.jl 2>&1

**Kết quả.** Toàn bộ 28.266 test đều đạt, không có test lỗi.

## 4. Minh họa FPClose qua dòng lệnh

**Ví dụ nhỏ.** CSDL `data/toy/pasquier99.txt` gồm 5 giao dịch, là ví dụ minh họa được sử dụng trong tài liệu của SPMF. Với minsup bằng 40%, chương trình xuất ra ba file:
- các tập phổ biến đóng (closed frequent itemsets);
- mọi tập phổ biến, được khôi phục từ các tập đóng (bước recovery);
- 10 luật kết hợp (association rules) có lift cao nhất trong số các luật có confidence ≥ 0,6.

Mỗi dòng output có dạng `1 2 3 #SUP: 4`, biểu thị itemset {1, 2, 3} xuất hiện trong 4 giao dịch. Thời gian thực thi của từng giai đoạn được in kèm theo.

**Dữ liệu thực tế.** Tiếp theo, chương trình được chạy theo đúng dạng lệnh `demo.jl --minsup <tỉ lệ> data/benchmark/<file>.txt` trên bộ Chess với minsup bằng 0,8.

In [ ]:
!mkdir -p out
!echo '== Input: data/toy/pasquier99.txt' && cat data/toy/pasquier99.txt
!julia --project=. demo.jl --minsup 40% --output-closed out/demo_closed.txt --output-all out/demo_all.txt --rules out/demo_rules.txt --minconf 0.6 --top 10 data/toy/pasquier99.txt

In [ ]:
import os
for title, path in [('Tập phổ biến đóng', 'out/demo_closed.txt'),
                    ('Mọi tập phổ biến', 'out/demo_all.txt'),
                    ('Luật kết hợp top-10 theo lift', 'out/demo_rules.txt')]:
    print('=' * 10, title, f'({path})')
    print(open(path, encoding='utf-8').read() if os.path.exists(path) else '(không có file)')

In [ ]:
%%time
# Chạy theo đúng dạng lệnh của đề: julia --project=. demo.jl --minsup <tỉ lệ> data/benchmark/<file>.txt
# Tập đóng được in ra stdout và chuyển vào file; thời gian từng giai đoạn hiển thị trong ô (stderr).
if ensure_data(['chess']):
    os.makedirs(os.path.join(REPO_DIR, 'out'), exist_ok=True)
    sh('julia --project=. demo.jl --minsup 0.8 data/benchmark/chess.txt > out/chess_0.8_closed.txt')
    out_path = os.path.join(REPO_DIR, 'out', 'chess_0.8_closed.txt')
    if os.path.exists(out_path):
        lines = [l for l in open(out_path, encoding='utf-8').read().splitlines() if '#SUP:' in l]
        print('Số tập phổ biến đóng (Chess, minsup 0,8):', len(lines))
        print('5 dòng đầu:')
        print('\n'.join(lines[:5]))

**Kết quả.** Trên `pasquier99.txt` (minsup 40%), chương trình tìm được 5 tập phổ biến đóng, trùng với ví dụ trong tài liệu SPMF. Trên Chess (minsup 0,8), chương trình tìm được 5083 tập phổ biến đóng; kết quả này được đối chiếu với SPMF tại mục 5.

## 5. Đối chiếu với SPMF

SPMF là thư viện Java có sẵn cài đặt FPClose và được nhóm sử dụng làm **cài đặt tham chiếu** để đánh giá tính đúng đắn. Mục này sử dụng hai bộ dữ liệu Chess và Mushrooms đã chuẩn bị ở mục 2 (nếu thiếu, dữ liệu sẽ được tải bổ sung tự động). Quá trình đối chiếu gồm hai bước:

1. **Kiểm tra công cụ tham chiếu** (`scripts/reference_check.jl`): Java 25 (Temurin) và `spmf.jar` được tải về thư mục `tools/`; sau khi kiểm tra mã băm của hai bộ dữ liệu, SPMF được chạy ở toàn bộ 7 mức minsup của mỗi bộ trong `data/meta/benchmark_config.csv` (14 cấu hình), và số tập đóng của SPMF được so với cột `reference_closed_itemsets`. Kết quả được ghi vào file riêng `results/reference/spmf_check_chess+mushrooms.csv`, không ghi đè kết quả đã lưu trong repository. Bước biên dịch cài đặt tham khảo FIMI được bỏ qua (tùy chọn `--skip-fimi`) nhằm rút ngắn thời gian thực thi.
2. **So sánh cài đặt của nhóm với SPMF** (`scripts/compare_spmf.jl`): từng itemset cùng support tương ứng được so sánh giữa hai cài đặt, trên các CSDL nhỏ đặt trực tiếp trong `data/toy/` (`pasquier99.txt`, `rounding_test.txt`) ở các mức minsup 60%, 40%, 20% và 7%, cho cả tập phổ biến đóng và mọi tập phổ biến (khôi phục từ tập đóng, so với `FPGrowth_itemsets` của SPMF), và trên tập phổ biến đóng của Chess ở minsup 80%. Hai cài đặt được xem là khớp hoàn toàn khi không tồn tại itemset chỉ xuất hiện ở một bên và không có itemset nào lệch support.

Giữa hai bước, hàm `ensure_java()` bổ sung Java 25 vào biến môi trường `PATH`, do `compare_spmf.jl` gọi trực tiếp lệnh `java` của hệ thống.

In [ ]:
ensure_data(['chess', 'mushrooms'])

In [ ]:
%%time
!julia --project=. scripts/reference_check.jl --datasets chess,mushrooms --skip-fimi

In [ ]:
need_py('pandas')
import os, pandas as pd
p = 'results/reference/spmf_check_chess+mushrooms.csv'
display(pd.read_csv(p)) if os.path.exists(p) else print('Chưa có', p)

In [ ]:
%%time
# Bảng khớp từng itemset trên mọi CSDL toy (closed và mọi tập phổ biến), sau đó một cấu hình của chess (80%)
ensure_java()
!julia --project=. scripts/compare_spmf.jl --table
!julia --project=. scripts/compare_spmf.jl data/benchmark/chess.txt 80%

**Kết quả.** Cài đặt của nhóm khớp hoàn toàn với SPMF về từng itemset và support trên cả 8 cấu hình của hai CSDL nhỏ trong `data/toy/` (tập đóng và mọi tập phổ biến) và trên Chess ở minsup 80% (5083/5083 tập đóng).

## 6. Thực nghiệm

### 6a. Tái hiện một phần thực nghiệm trên Colab

Nhằm minh họa quy trình đo, `scripts/run_benchmark_grid.jl` được chạy ở chế độ rút gọn: chỉ trên bộ Chess, với 3 mức minsup (90%, 75% và 65%: mức cao nhất, giữa và gần cuối của lưới) và mỗi cấu hình đo một lần, gồm phần kiểm tra tính đúng đắn và phần đo thời gian so với SPMF. Các file kết quả mang hậu tố `_quick`, vì vậy không ghi đè kết quả chính thức.

Số đo trên Colab chỉ mang tính minh họa do phụ thuộc vào cấu hình máy ảo. Số liệu chính thức của báo cáo là kết quả đo đầy đủ trên máy cá nhân của nhóm, được trình bày tại mục 6b.

In [ ]:
%%time
if ensure_data(['chess']):
    ensure_java()
    sh(f'julia --project=. scripts/run_benchmark_grid.jl --quick --datasets chess --only correctness,runtime --machine {MACHINE_LABEL}')

In [ ]:
need_py('pandas')
import os, pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
for p in ['results/correctness/grid_correctness_quick.csv',
          'results/runtime/grid_runtime_summary_quick.csv']:
    print('==', p)
    display(pd.read_csv(p)) if os.path.exists(p) else print('  (chưa có file)')

**Kết quả.** Quy trình đo hoạt động đầy đủ trên Colab; cả 3 cấu hình của Chess (minsup 90%, 75%, 65%) đều khớp với SPMF. 

### 6b. Kết quả đo chính thức

Số liệu Chương 2 trong báo cáo được đo trên máy cá nhân của nhóm, không phải trên Colab; cấu hình phần cứng và phần mềm của từng máy được ghi tại `results/ENVIRONMENT.md` và `results/ENVIRONMENT_hyt.md`. Quy ước đo như sau:
- mỗi cấu hình được đo 3 lần, với giới hạn 600 giây cho mỗi lần chạy; các bảng trong `results/summary/` và các hình lấy trung vị của các lần chạy thành công (các file `*_summary.csv` do `run_benchmark_grid.jl` ghi lấy giá trị nhỏ nhất, như ghi trong `results/ENVIRONMENT.md`);
- khi so sánh với SPMF, thời gian của cả hai cài đặt đều bao gồm bước đọc dữ liệu và ghi file kết quả;
- bộ nhớ của nhóm là mức tăng bộ nhớ đỉnh (peak RSS) so với mức nền của Julia; bộ nhớ của SPMF là heap của JVM, do đó không được so sánh trực tiếp với số liệu của nhóm.

Các bảng tổng hợp được sinh bởi `scripts/summarize_results.jl` và các hình được vẽ bởi `scripts/plot_results.jl`. Phần phân tích chi tiết được trình bày tại Chương 2 của báo cáo.

In [ ]:
print(open('results/ENVIRONMENT.md', encoding='utf-8').read())

In [ ]:
need_py('pandas')
import os, pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
SUMMARY = 'results/summary'

def show(name, title, cols=None):
    path = f'{SUMMARY}/{name}.csv'
    if not os.path.exists(path):
        print('Chưa có', path)
        return
    df = pd.read_csv(path)
    if cols:
        df = df[[c for c in cols if c in df.columns]]
    print(f'== {title} ({path}, {len(df)} dòng)')
    display(df)

show('correctness_summary', 'Tính đúng đắn: cài đặt của nhóm so với SPMF')
show('speedup', 'Thời gian: cài đặt của nhóm so với SPMF (tỷ số SPMF / nhóm > 1 nghĩa là nhóm nhanh hơn)')
show('time_breakdown', 'Phân rã thời gian của cài đặt nhóm')
show('ablation', 'Ablation các tối ưu (% so với optimized)',
     cols=['dataset', 'minsup_pct', 'variant', 'n_closed', 'mine_s', 'mine_s_vs_opt_pct',
           'mem_delta_mb', 'mem_delta_mb_vs_opt_pct', 'alloc_mb', 'alloc_mb_vs_opt_pct'])
show('scalability', 'Khả năng mở rộng (Retail 10%–100%)')
show('txlen', 'Ảnh hưởng độ dài giao dịch')

In [ ]:
import os
from IPython.display import Image, Markdown, display
FIG = 'results/figures'
figures = [
    ('runtime_grid', 'Thời gian chạy theo minsup trên 9 bộ dữ liệu (nhóm và SPMF)', 900),
    ('itemsets_vs_minsup', 'Số tập phổ biến đóng theo minsup', 900),
    ('time_breakdown', 'Phân rã thời gian của cài đặt nhóm', 600),
    ('memory_variants_grid', 'Bộ nhớ thực dùng theo biến thể', 900),
    ('time_variants_grid', 'Thời gian khai thác theo biến thể', 900),
    ('ablation_time_relative', 'Ablation thời gian (so với optimized)', 600),
    ('scalability_time', 'Khả năng mở rộng theo kích thước', 600),
    ('txlen_time', 'Thời gian theo độ dài giao dịch', 600),
    ('txlen_itemsets', 'Số tập đóng theo độ dài giao dịch', 600),
]
for name, caption, width in figures:
    path = f'{FIG}/{name}.png'
    if os.path.exists(path):
        display(Markdown(f'**{caption}** (`{path}`)'))
        display(Image(filename=path, width=width))
    else:
        print('Chưa có', path)

**Nhận xét.** Cài đặt của nhóm khớp 100% với SPMF trên toàn bộ 56 cấu hình thuộc 9 bộ benchmark. Về thời gian, cài đặt của nhóm nhanh hơn SPMF ở 47/56 cấu hình (trung bình nhân khoảng 1,72 lần), với ưu thế rõ nhất trên dữ liệu dày đặc (trung bình nhân 3,85 lần trên Chess và 2,99 lần trên Mushrooms); trên T10I4D100K, tỷ số SPMF/nhóm chỉ là 0,84. Phân tích chi tiết được trình bày tại Chương 2 của báo cáo.

## 7. Ứng dụng thực tế (Chương 3)

Notebook này tái lập kết quả chính của mỗi ứng dụng bằng cài đặt FPClose của nhóm; quy trình từng bước được trình bày trong `notebooks/application_01.ipynb` và `notebooks/application_02.ipynb`. Toàn bộ itemset và luật kết hợp đều được sinh bằng chính cài đặt FPClose của nhóm, thông qua dòng lệnh `demo.jl` hoặc các hàm trong `src/` (`read_spmf`, `fpclose`, `recover_all_frequent`, `generate_rules`, ...). Các thư viện xử lý ảnh, phân cụm k-means và pandas chỉ được dùng ở bước biến đổi dữ liệu thành giao dịch và trình bày kết quả.

### 7.1 Ứng dụng 1 – Phân tích giỏ hàng (market basket)

**Mục tiêu.** Xác định các nhóm mặt hàng thường được mua cùng nhau nhằm hỗ trợ gợi ý sản phẩm và bán chéo. Cụ thể, các tập mặt hàng phổ biến đóng được khai thác bằng FPClose, từ đó sinh các luật kết hợp $X \rightarrow Y$ thỏa $\mathrm{sup} \ge \mathit{minsup}$ và $\mathrm{conf} \ge \mathit{minconf}$, rồi xếp hạng theo $\mathrm{lift}(X \rightarrow Y) = \mathrm{conf}(X \rightarrow Y) / \mathrm{sup}(Y)$.

**Dữ liệu.** UCI Online Retail [1] (giấy phép CC BY 4.0): 541.909 dòng dữ liệu, mỗi dòng là một mặt hàng trong một hóa đơn, của một nhà bán lẻ trực tuyến tại Vương quốc Anh, giai đoạn 12/2010–12/2011.

**Tiền xử lý** (`applications/application_01/market_basket.py`):
1. Tải file gốc từ UCI và kiểm tra mã băm SHA-256 để bảo đảm tính tái lập.
2. Loại các dòng thiếu InvoiceNo, StockCode hoặc Description; hóa đơn hủy; dòng có Quantity ≤ 0 hoặc UnitPrice ≤ 0; và các mã không phải sản phẩm (phí vận chuyển, điều chỉnh kế toán, phiếu quà tặng). Dòng thiếu CustomerID được giữ lại vì giao dịch được xác định theo InvoiceNo.
3. Mỗi hóa đơn là một giao dịch, gồm tập các StockCode phân biệt: bỏ qua số lượng, gộp mã trùng, giữ cả hóa đơn chỉ có một sản phẩm.
4. Ánh xạ StockCode sang số nguyên và ghi theo định dạng SPMF.

**Tham số và đầu ra.** $\mathit{minsup} = 1\%$, $\mathit{minconf} = 0{,}6$, $k = 10$. Các ngưỡng được chọn qua khảo sát minsup trong [`notebooks/application_01.ipynb`](https://colab.research.google.com/github/DRAGON-1312/FPClose/blob/main/notebooks/application_01.ipynb); notebook này cũng trình bày chẩn đoán dữ liệu, ảnh hưởng của hóa đơn rất dài và phân tích độ nhạy. Mục này báo cáo hai bảng: top-$k$ luật theo lift, và top-$k$ luật đa dạng (các luật đôi một không có mặt hàng chung).

[1] D. Chen, *Online Retail* [Dataset], UCI Machine Learning Repository, 2015. DOI: [10.24432/C5BW33](https://doi.org/10.24432/C5BW33)

In [ ]:
%%time
# Tiền xử lý: UCI Online Retail -> giao dịch định dạng SPMF, bằng các hàm có sẵn của market_basket.py.
need_py('pandas', 'openpyxl', 'jinja2')   # jinja2: cần cho pandas Styler (mb.styled)
import sys, time
from pathlib import Path
import pandas as pd

ROOT = Path(REPO_DIR)
APP = ROOT / 'applications' / 'application_01'
if str(APP) not in sys.path:
    sys.path.insert(0, str(APP))
import market_basket as mb
mb = importlib.reload(mb)

MINSUP, MINCONF, TOP_K = 0.01, 0.6, 10    # giống application_01.ipynb
OUT_DIR = APP / 'out'                     # output thô của FPClose (không commit)
RULE_COLS = ['rank_all', 'antecedent', 'consequent', 'support', 'confidence', 'lift', 'count']

t0 = time.perf_counter()
df_raw = mb.load_raw(APP, mb.UCI_URL, mb.RAW_SHA256)              # tải + kiểm SHA-256 (lần đầu ~23 MB)
df_clean, cleaning_steps = mb.clean(df_raw)                        # mặc định NON_PRODUCT, NON_PRODUCT_PREFIXES
transactions, item_map, tx_info = mb.to_transactions(df_clean)     # id từ ID_START = 1
spmf_files = mb.write_spmf(transactions, item_map, APP)            # data/transactions.txt, data/item_map.csv
tx_stats, top10_items = mb.transaction_stats(transactions, item_map)
print(f"{len(df_raw):,} dòng gốc -> {len(df_clean):,} dòng sau làm sạch -> {tx_info['n_transactions']:,} giao dịch "
      f"trên {tx_info['n_items']:,} item ({time.perf_counter() - t0:.1f} s)")
display(mb.styled(tx_stats).format({'giá trị': '{:g}'}))

In [ ]:
%%time
# Khai thác bằng FPClose của nhóm (demo.jl): tập đóng, mọi tập phổ biến và TOÀN BỘ luật ở (MINSUP, MINCONF).
t0 = time.perf_counter()
run = mb.run_fpclose(ROOT, spmf_files['transactions'], MINSUP, OUT_DIR, minconf=MINCONF, timeout=600)
info = run['info']
all_rules = mb.read_rules(run['rules'], item_map)
assert info.get('transactions') == len(transactions), 'số giao dịch demo.jl đọc khác số giao dịch đã ghi'
assert len(all_rules) == info.get('rules_generated', len(all_rules)), 'số luật đọc được khác số demo.jl báo'
print(f"Số giao dịch: {info.get('transactions')} | minsup = {MINSUP} ({info.get('minsup_count')} giao dịch), "
      f"minconf = {MINCONF}")
print(f"Số tập phổ biến đóng: {info.get('closed_itemsets')} | số tập phổ biến: {info.get('frequent_itemsets')} | "
      f"số luật: {len(all_rules)}")
print(f"Thời gian: khai thác {info.get('mining_seconds')} s, khôi phục {info.get('recovery_seconds')} s, "
      f"cả tiến trình (gồm khởi động Julia) {run['wall_s']:.2f} s")

# Kiểm tra chéo độc lập với FPClose: tính lại support, confidence, lift của mọi luật bằng phép đếm trực tiếp.
rule_check = mb.verify_rules(all_rules, transactions)
print(f"Kiểm tra chéo: {int(rule_check['khớp'].sum())}/{len(rule_check)} luật khớp với phép đếm trực tiếp")
assert rule_check['khớp'].all(), 'Có luật không khớp với phép đếm trực tiếp'

top10_rules = all_rules[all_rules['rank'] <= TOP_K].reset_index(drop=True)   # file luật đã xếp theo lift như --top
top10_rules['rank_all'] = top10_rules['rank']
top10_diverse = mb.diverse_top_rules(all_rules, TOP_K, mode='item')
print(f'\nTop-{TOP_K} luật theo lift:')
display(mb.styled(top10_rules[RULE_COLS]))
print(f'Top-{TOP_K} luật đa dạng (không có sản phẩm chung giữa các luật; rank_all là thứ hạng trong toàn bộ luật):')
display(mb.styled(top10_diverse[RULE_COLS]))
print(f'({time.perf_counter() - t0:.1f} s)')

In [ ]:
# Đối chiếu tái lập: hai bảng vừa sinh so với kết quả đã commit của application_01.ipynb.
# antecedent, consequent, count phải khớp tuyệt đối; support, confidence, lift lệch < 1e-9.
def matched_rows(new, ref):
    if len(new) != len(ref):
        return 0
    ok = 0
    for a, b in zip(new.itertuples(index=False), ref.itertuples(index=False)):
        ok += (a.antecedent == b.antecedent and a.consequent == b.consequent and int(a.count) == int(b.count)
               and all(abs(float(getattr(a, c)) - float(getattr(b, c))) < 1e-9
                       for c in ('support', 'confidence', 'lift')))
    return ok

for name, new, fname in [('Top-10 theo lift', top10_rules, 'top10_rules.csv'),
                         ('Top-10 đa dạng', top10_diverse, 'top10_rules_diverse.csv')]:
    ref = pd.read_csv(APP / 'results' / fname)
    k = matched_rows(new, ref)
    print(f'{name}: khớp {k}/{len(ref)} với applications/application_01/results/{fname}')
    assert len(ref) == TOP_K and k == len(ref), f'{name} không khớp với {fname}'

**Nhận xét.**
- **Dữ liệu:** sau tiền xử lý còn 19.773 giao dịch trên 3.791 mặt hàng. Dữ liệu rất thưa (mật độ 0,69%) và độ dài giao dịch lệch phải: trung vị 15 nhưng trung bình 26,2, dài nhất 1.106 sản phẩm; 236 hóa đơn (1,2%) có từ 200 sản phẩm trở lên.
- **Khai thác:** ở minsup 1% (198 giao dịch), FPClose tìm được 2.026 tập phổ biến đóng; số tập phổ biến khôi phục được cũng là 2.026, tức ở ngưỡng này mọi tập phổ biến đều đóng. Từ đó sinh được 574 luật với minconf 0,6. Toàn bộ 574 luật khớp với phép đếm trực tiếp và hai bảng top-10 khớp với kết quả đã lưu trong `results/`, cho thấy kết quả tái lập được.
- **Top-10 theo lift:** cả mười luật đều thuộc bộ que cắm cây gia vị (Herb Marker: Thyme, Rosemary, Parsley, Basil), với lift 76,8–78,8 và confidence 0,83–0,95. Support chỉ 1,01–1,12% (200–221 hóa đơn), sát ngưỡng minsup. Các luật xuất hiện theo cặp đối xứng có cùng lift, do lift đối xứng theo định nghĩa.
- **Top-10 đa dạng:** khi yêu cầu các luật đôi một không có mặt hàng chung, bảng gồm các dòng que cắm cây gia vị, đồ Regency, đồ trang trí Giáng sinh bằng gỗ, bộ đồ chơi Poppy's Playhouse, hũ mứt, cốc và bát chấm bi, bộ dao nĩa trẻ em, với lift 38,4–78,8 và confidence 0,65–0,87. Điểm chung là **khách hàng mua nhiều biến thể của cùng một bộ sưu tập**; đây là cơ sở cho chiến lược gợi ý sản phẩm theo hướng "hoàn thiện bộ sưu tập" và lập kế hoạch tồn kho theo bộ sưu tập.
- **Diễn giải:** lift đo mức đồng xuất hiện, không thể hiện quan hệ nhân quả hay thứ tự mua hàng; hiệu quả của các khuyến nghị cần được kiểm chứng bằng thử nghiệm (ví dụ A/B) trước khi triển khai.
- **Hạn chế:** support của các luật hàng đầu sát ngưỡng nên kết quả nhạy với lựa chọn minsup và với nhóm hóa đơn rất dài; mỗi biến thể là một StockCode riêng nên các luật chủ yếu nối các biến thể cùng dòng; số lượng, đơn giá và thời gian mua không được sử dụng; dữ liệu chỉ thuộc một cửa hàng tại Vương quốc Anh trong giai đoạn 2010–2011. Phân tích độ nhạy theo minsup và ảnh hưởng của hóa đơn rất dài được trình bày trong `notebooks/application_01.ipynb`.

### 7.2 Ứng dụng 2 – Khai thác mẫu trực quan trên ảnh

**Mục tiêu:** tìm các tổ hợp visual words phổ biến trong từng lớp ảnh,
phân tích chúng nằm trên vật thể hay nền và thử sử dụng làm đặc trưng phân loại.

**Dữ liệu:** Caltech-101, chọn 5 lớp: airplanes, Motorbikes, Faces_easy,
watch và Leopards; 200 ảnh/lớp. Chia 140 ảnh train và 60 ảnh test mỗi lớp,
seed 42.

**Biến đổi ảnh thành giao dịch:**
1. Chuyển ảnh sang mức xám, resize cạnh dài về 400 pixel.
2. Trích SIFT; học codebook bằng MiniBatchKMeans với k = 100 và 300,
   chỉ dùng descriptor từ train.
3. Gán descriptor vào visual word gần nhất. Mỗi ảnh trở thành tập các từ
   xuất hiện ít nhất 3 lần; loại từ có mặt trong hơn 80% ảnh train.
4. Xuất giao dịch SPMF, mỗi dòng một ảnh, item = mã visual word + 1.

**Khai thác:** chạy FPClose Julia của nhóm riêng trên giao dịch train
của từng lớp. Minsup được chọn là 55% với k = 100 và 25% với k = 300.

**Phạm vi demo bên dưới:** tải giao dịch k = 300 của lớp airplanes và
chạy lại FPClose ở minsup 25%. Đây là lớp có output nhỏ, phù hợp minh họa.
Kết quả tham chiếu có 132 tập phổ biến đóng, trong đó 102 tập dài từ hai item.
Bảng hiển thị 10 tập đóng dài từ hai item có support cao nhất; đây chưa phải top mẫu theo growth rate.

**Notebook đầy đủ:** gồm tải ảnh gốc, tạo giao dịch, khai thác cả năm lớp,
growth rate, minh họa keypoint và so sánh ba cấu hình phân loại:
[Ứng dụng 2 trên Colab](https://colab.research.google.com/github/DRAGON-1312/FPClose/blob/main/applications/app2_image/app2_image.ipynb).

**Dữ liệu trung gian:**
[Hugging Face](https://huggingface.co/datasets/nnkhlh376/fpclose-app2-caltech).
Demo này dùng giao dịch đã chuẩn bị; không thực hiện lại bước học codebook.

*Ô dưới gọi `julia`: cần chạy mục 1 (cài Julia) trước nếu không dùng "Run all".*

In [ ]:
from pathlib import Path
import hashlib
import subprocess
import urllib.request
import pandas as pd

app2_demo = Path(REPO_DIR) / "out" / "app2_demo"
app2_demo.mkdir(parents=True, exist_ok=True)

# Cố định commit dữ liệu trên Hugging Face (không dùng main), để số tập đóng kiểm tra bên dưới không đổi
# nếu sau này kho được cập nhật.
HF_REVISION = "48ff7f3dc503bfc3d2682163e1a4119c76d15980"
base_url = (
    "https://huggingface.co/datasets/"
    f"nnkhlh376/fpclose-app2-caltech/resolve/{HF_REVISION}"
)
relative_path = "transactions/k300_word/class_airplanes.txt"
input_file = app2_demo / "class_airplanes.txt"
output_file = app2_demo / "airplanes_closed.txt"

# Kiểm tra dữ liệu tải về bằng manifest SHA-256.
with urllib.request.urlopen(f"{base_url}/SHA256SUMS.txt") as response:
    manifest = response.read().decode("utf-8")

checksums = {}
for line in manifest.splitlines():
    if line.strip():
        digest, filename = line.split(maxsplit=1)
        filename = filename.lstrip("*").removeprefix("./")
        checksums[filename] = digest.lower()

expected_hash = checksums[relative_path]
with urllib.request.urlopen(f"{base_url}/{relative_path}") as response:
    data = response.read()

assert hashlib.sha256(data).hexdigest() == expected_hash, "Sai SHA-256"
input_file.write_bytes(data)

transactions = [
    list(map(int, line.split()))
    for line in data.decode("utf-8").splitlines()
    if line.strip()
]
assert len(transactions) == 140
print("Số giao dịch train:", len(transactions))
print("Độ dài trung bình:",
      round(sum(map(len, transactions)) / len(transactions), 2))

# Chạy chính FPClose của nhóm; dừng ô nếu lệnh thất bại.
subprocess.run(
    [
        "julia", f"--project={REPO_DIR}",
        str(Path(REPO_DIR) / "demo.jl"),
        "--minsup", "25%",
        "--output-closed", str(output_file),
        str(input_file),
    ],
    cwd=REPO_DIR,
    check=True,
)

rows = []
for line in output_file.read_text(encoding="utf-8").splitlines():
    if line.strip():
        items, support = line.split("#SUP:")
        rows.append({
            "items": items.strip(),
            "length": len(items.split()),
            "support_count": int(support),
            "support_relative": int(support) / len(transactions),
        })

patterns = pd.DataFrame(rows)
print("Số tập đóng:", len(patterns))
print("Số tập đóng dài ≥ 2:", int((patterns["length"] >= 2).sum()))
assert len(patterns) == 132
assert (patterns["length"] >= 2).sum() == 102

# Hiển thị tổ hợp (dài >= 2), support giảm dần; chưa phải xếp hạng theo growth rate.
display(patterns[patterns["length"] >= 2]
        .sort_values(["support_count", "length"], ascending=False)
        .head(10))

**Nhận xét.** Demo tái khai thác các tập phổ biến đóng từ giao dịch visual
words bằng FPClose của nhóm. Support cho biết số ảnh train chứa trọn tập
từ, chưa cho biết mẫu đặc trưng riêng cho lớp hay nằm trên vật thể.

Trong thí nghiệm đầy đủ, nhóm tính growth rate bằng support tương đối
trong và ngoài lớp, rồi đối chiếu keypoint với contour vật thể.
Các mẫu được chọn của Leopards tập trung trên lông báo; một số mẫu của
Faces_easy và Motorbikes có nhiều keypoint ở nền hoặc biên ảnh.
GR cao vì vậy không nhất thiết đồng nghĩa với đặc trưng của vật thể.

Với k = 300, accuracy của BoVW, đặc trưng mẫu và biểu diễn kết hợp lần
lượt là 87,00%, 60,33% và 87,67% trên 300 ảnh test.
Đây là kết quả của thí nghiệm đầy đủ trong notebook riêng, không phải
kết quả được tính lại bởi ô demo phía trên. Cách biểu diễn mẫu hiện tại
chưa cho thấy lợi ích phân loại rõ ràng; giá trị chính là hỗ trợ phân tích
các mẫu và thông tin nền/biên ảnh.

## 8. Kết luận

**Tính đúng đắn.** 28.266/28.266 unit test đều đạt, gồm cả đối chiếu với đáp án vét cạn. Kết quả của nhóm khớp SPMF về từng itemset và support trên 56/56 cấu hình của 9 bộ benchmark và trên các CSDL nhỏ trong `data/toy/`.

**Hiệu năng.**
- Cài đặt của nhóm nhanh hơn SPMF ở 47/56 cấu hình, trung bình nhân 1,72 lần.
- Tắt toàn bộ tối ưu (`basic`) làm thời gian khai thác tăng trung bình nhân 10,4% (tối đa 44,4%); tắt riêng từng tối ưu chỉ tăng 0–3,9%.
- Trên các tập con 10%–100% của Retail (minsup 0,1%), thời gian tăng từ 0,07 s lên 0,47 s.
- Khi độ dài giao dịch trung bình tăng từ 5 lên 30 (minsup 0,2%), số tập đóng tăng từ 914 lên 146.134, tỷ số SPMF/nhóm giảm từ 4,14 xuống 1,56.

**Ứng dụng.** Trên UCI Online Retail, FPClose sinh 574 luật, đều khớp với phép đếm trực tiếp; các luật nổi bật cho thấy khách hàng thường mua nhiều biến thể của cùng một bộ sưu tập. Trên Caltech-101, các tập visual word đóng hỗ trợ phân tích mẫu của từng lớp nhưng chưa cải thiện rõ phân loại (87,67% so với 87,00% của BoVW, k = 300).

**Hạn chế và hướng phát triển.**
- Trên dữ liệu thưa với minsup thấp, cài đặt của nhóm có thể chậm hơn SPMF (tỷ số 0,84 trên T10I4D100K); đây là hướng tối ưu tiếp theo.
- Luật kết hợp chỉ phản ánh đồng xuất hiện; các khuyến nghị cần được kiểm chứng bằng thử nghiệm A/B.

In [ ]:
print(f'Tổng thời gian chạy notebook: {(time.time() - T_START) / 60:.1f} phút')